# Training A Surrogate Model with PyTorch

This notebook illustrates how to train a simple neural network surrogate model using Pytorch. In this notebook we look at the simple case of the `arnett_with_features` function from redback. We show how to generate the training data, fit the model, and evaluate the model.

This notebook uses multiple packages that are not installed with redback_surrogates by default (and must be install manually), including:

   * redback (for the arnett model)
   * torch (for the training)

In [1]:
import numpy as np

from astropy.table import Table
from scipy.stats import loguniform
from redback.transient_models.supernova_models import arnett_with_features

from redback_surrogates.learned_surrogate import LearnedSurrogateModel
from redback_surrogates.learned_surrogate_data import LearnedSurrogateDataset
from redback_surrogates.learned_surrogate_train import evaluate_learned_model, train_pytorch_model

No module named 'lalsimulation'
lalsimulation is not installed. Some EOS based models will not work. Please use bilby eos or pass your own EOS generation class to the model
09:05 bilby INFO    : Running bilby version: 2.7.1
09:05 redback INFO    : Running redback version: 1.12.1


## Creating the Training Data

We start with a simple method to create the training data. This consists of three steps:
   1) We generate all of the input parameters needed by the function.
   2) Generate an output grid (spectra) for each of the sets of parameters.
   3) Wrap the data in a `LearnedSurrogateDataset`.

We put these steps in a helper function so we can use the same approach for generating the test data.

In [2]:
def generate_arnett_data(num_samples):
    """Generate training/validation data for the Arnett model with
    randomly sampled parameters and outputs as the spectra.

    :param num_samples: Number of samples to generate.
    """
    times = np.array([0.0, 1.0])  # Unused by "spectra" output type

    # Generate the parameter samples.
    data = {
        "redshift": loguniform(0.001, 3.0).rvs(size=num_samples),
        "f_nickel": loguniform(0.001, 1.0).rvs(size=num_samples),
        "mej": loguniform(0.0001, 100.0).rvs(size=num_samples),
        "vej": loguniform(1_000, 100_000).rvs(size=num_samples),
        "kappa": loguniform(0.0001, 10000.0).rvs(size=num_samples),
        "kappa_gamma": loguniform(0.01, 0.1).rvs(size=num_samples),
        "temperature_floor": loguniform(1_000, 100_000).rvs(size=num_samples),
    }
    
    # Compute the output for each sample.
    output_grids = []
    out_times = None
    out_lambdas = None
    for i in range(num_samples):
        grid = arnett_with_features(
            time=times,
            redshift=data["redshift"][i],
            f_nickel=data["f_nickel"][i],
            mej=data["mej"][i],
            vej=data["vej"][i],
            kappa=data["kappa"][i],
            kappa_gamma=data["kappa_gamma"][i],
            temperature_floor=data["temperature_floor"][i],
            output_format="spectra",
        )
        output_grids.append(grid.spectra)

        # Save the time and wavelength grid from the first sample.
        if i == 0:
            out_times = grid.time
            out_lambdas = grid.lambdas

    data["grid"] = np.array(output_grids)

    results = LearnedSurrogateDataset(
        Table(data),
        output_column="grid",
        times=out_times,
        wavelengths=out_lambdas,
    )
    return results

In [3]:
training_data = generate_arnett_data(num_samples=100)

In [4]:
model = train_pytorch_model(
    training_data,
    hidden_sizes=[128, 64, 128],
    training_epochs=10,
    verbose=True,
)

Starting training...


100%|██████████| 10/10 [00:01<00:00,  6.60it/s]


Epoch 10/10, Loss: 10.562071992701268


W1208 09:05:55.529000 3260 torch/onnx/_internal/exporter/_registration.py:107] torchvision is not installed. Skipping torchvision::nms


[torch.onnx] Obtain model graph for `NormalizedMultilevelSigmoid([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NormalizedMultilevelSigmoid([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


2025-12-08 09:05:55,910 - onnx_ir.passes.common.unused_removal - INFO - Removed 8 unused nodes
2025-12-08 09:05:55,910 - onnx_ir.passes.common.unused_removal - INFO - No unused functions to remove
2025-12-08 09:05:55,913 - onnxscript.optimizer._constant_folding - INFO - Skipping constant folding for node 'node_view' because it is graph input to preserve graph signature
2025-12-08 09:05:55,913 - onnxscript.optimizer._constant_folding - INFO - Skipping constant folding for node 'node_view_1' because it is graph input to preserve graph signature
2025-12-08 09:05:55,914 - onnxscript.optimizer._constant_folding - INFO - Skipping constant folding for node 'node_view_2' because it is graph input to preserve graph signature
2025-12-08 09:05:55,914 - onnxscript.optimizer._constant_folding - INFO - Skipping constant folding for node 'node_view_3' because it is graph input to preserve graph signature
2025-12-08 09:05:55,915 - onnxscript.optimizer._constant_folding - INFO - Skipping constant foldi

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


2025-12-08 09:05:56.123 Python[3260:51142] 2025-12-08 09:05:56.123575 [W:onnxruntime:, constant_folding.cc:278 ApplyImpl] Could not find a CPU kernel and hence can't constant fold CastLike node 'n3'
2025-12-08 09:05:56.123 Python[3260:51142] 2025-12-08 09:05:56.123900 [W:onnxruntime:, constant_folding.cc:278 ApplyImpl] Could not find a CPU kernel and hence can't constant fold CastLike node 'n3'


In [5]:
test_data = generate_arnett_data(num_samples=20)

mse = evaluate_learned_model(model, test_data)
print(f"Test MSE: {mse}")

TypeError: 'LearnedSurrogateModel' object is not callable